# RealMLP v2 1단계 스크리닝 — 싼-배깅 레시피 (ep64×n_ens4, fold0)

docs/wiki/realmlp_v2_plan.md 1단계. exp_031_rmlp_v2_s1.  
비교기준: exp_024 fold0 = 0.949893. kill_criterion: fold0 >= 0.9492.

In [ ]:
# 1) input 자동탐색 (마운트 비표준: /kaggle/input/{datasets,competitions}/...) — torch import 前
import sys, os, glob, subprocess
from pathlib import Path
print('/kaggle/input:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'NONE')
c = glob.glob('/kaggle/input/**/src/config.py', recursive=True); assert c, 'src/config.py 못 찾음'
SRC_ROOT = str(Path(c[0]).parents[1]); print('SRC_ROOT:', SRC_ROOT)
cc = glob.glob('/kaggle/input/**/playground-series-s6e5', recursive=True); assert cc, '대회 폴더 못 찾음'
COMP = Path(cc[0]); print('COMP:', COMP)
ac = glob.glob('/kaggle/input/**/f1_strategy_dataset*.csv', recursive=True); assert ac, '증강 csv 못 찾음'
AUG = Path(ac[0]); print('AUG:', AUG)

In [ ]:
# 2) GPU 종류 감지(nvidia-smi, torch import 前) → 조건부 torch. 그 위에 프로젝트 deps.
GPU = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', GPU)
def pip(*a): subprocess.run([sys.executable,'-m','pip','install','-q',*a], check=True)
if 'P100' in GPU:
    print('P100(sm_60) → cu121 torch trio 재설치 (Kaggle 기본 torch 는 sm_70+ 만)')
    pip('torch==2.5.1','torchvision==0.20.1','torchaudio==2.5.1',
        '--index-url','https://download.pytorch.org/whl/cu121')
else:
    print('T4 등(sm_75+) → Kaggle 기본 torch 유지')
pip('pytabkit','hydra-core','python-dotenv')

In [ ]:
# 3) torch CUDA 실연산 검증 + import 체인 fast-fail
import torch
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU', torch.cuda.get_device_name(0))
_x = torch.randn(256, 256, device='cuda'); _v = (_x @ _x).sum().item()
print('CUDA matmul OK')
sys.path.insert(0, SRC_ROOT)
from src import config
from src.train_realmlp import run
print('import OK:', config.__file__)

In [ ]:
# 4) 경로 override
config.TRAIN_PATH = COMP / 'train.csv'
config.TEST_PATH = COMP / 'test.csv'
config.SAMPLE_SUBMISSION_PATH = COMP / 'sample_submission.csv'
config.SOURCE_AUG_PATH = AUG
out = Path('/kaggle/working')
config.OOF_DIR = out / 'oof'; config.SUBMISSION_DIR = out / 'submissions'; config.LOG_DIR = out / 'logs'
import pandas as pd
_a = pd.read_csv(config.SOURCE_AUG_PATH); print('AUG shape:', _a.shape)
assert len(_a) == 101371, f'증강 행수 불일치: {len(_a)}'
assert config.TRAIN_PATH.exists(), f'train.csv 없음: {config.TRAIN_PATH}'

In [ ]:
# 5) cfg 구성 — ep64 × n_ens4 (비용 ≈ 256ep×1), max_folds=1 (fold0 스크리닝)
from omegaconf import OmegaConf
CONF = Path(SRC_ROOT) / 'conf'
model_cfg = OmegaConf.load(CONF / 'model' / 'realmlp.yaml')
# n_epochs, n_ens override (기본값 유지 후 덮어쓰기)
model_cfg.params.n_epochs = 64
model_cfg.params['n_ens'] = 4
cfg = OmegaConf.create({
    'exp_id': 'exp_031_rmlp_v2_s1',
    'notes': 'v2 stage1 screening: ep64 x n_ens4 fold0 vs exp_024_fold0=0.949893',
    'use_wandb': False,
    'max_folds': 1,
    'model': model_cfg,
    'features': OmegaConf.load(CONF / 'features' / 'realmlp_fe_yearcat.yaml'),
    'augment': {'enabled': True, 'weight': 1.0},
})
print(OmegaConf.to_yaml(cfg))
print('kill_criterion: fold0 >= 0.9492 → 2단계 진행')

In [ ]:
# 6) 학습 (fold0만). 예상 ~5분 (ep64×n_ens4).
import time
t0 = time.time()
result = run(cfg)
print(result, f'\n총 {time.time()-t0:.0f}s')

In [ ]:
# 7) 산출물 확인 + kill_criterion 판정
fold0_auc = result.get('fold_aucs', [None])[0]
print(f'fold0 AUC = {fold0_auc}')
print(f'exp_024 fold0 = 0.949893')
if fold0_auc is not None:
    diff = fold0_auc - 0.949893
    print(f'diff = {diff:+.6f}')
    print('PASS → 2단계 진행' if fold0_auc >= 0.9492 else 'FAIL → 싼-배깅 regime 포기 또는 lr 스윕')
oof_path = out / 'oof' / 'exp_031_rmlp_v2_s1.csv'
if oof_path.exists():
    oof = pd.read_csv(oof_path)
    print('OOF:', oof.shape, list(oof.columns))